# HookLink2iOS — Ocarina of Time

Notebook no mesmo formato simples do Link2iOS original: autenticação, checkout, recuperação, preparação, handoff e retorno da IPA ao Drive.

**Origem:** cole diretamente o link HTTPS do arquivo. O Colab identifica NUP, RAR, ZIP, 7Z ou tar compactado. Na primeira execução baixa e, se necessário, extrai; salva o NUP no Google Drive. Nas próximas execuções, procura esse NUP no Drive antes de baixar novamente.

**Destino:** branch HookLink2iOS, Title ID 0400486024DEC000, versão v131072, nxnative fixado, build iOS em macos-26, romfs separate.

Reutiliza a pasta raiz e o OAuth do Link2iOS original. Não pede outro ID de pasta nem usa service account.

In [ ]:
!pip -q install requests google-api-python-client google-auth pynacl
!apt-get update -qq
!apt-get install -y -qq libarchive-tools

import os, sys, json, time, hashlib, subprocess, importlib
from pathlib import Path
from getpass import getpass
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

REPO = "joao9vegeto-stack/KintTany"
BRANCH = "HookLink2iOS"
DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
ROOT = Path("/content/hooklink2ios-work")

auth.authenticate_user()
DRIVE_CREDENTIALS, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive_api_client = build("drive", "v3", credentials=DRIVE_CREDENTIALS, cache_discovery=False)
folder = drive_api_client.files().get(fileId=DRIVE_FOLDER_ID, fields="id,name,mimeType").execute()
assert folder.get("mimeType") == "application/vnd.google-apps.folder", folder
print("Google Drive autenticado:", folder.get("name"))

SOURCE_URL = getpass("Cole a URL HTTPS completa do .nup/.rar/.zip/.7z: ").strip()
assert SOURCE_URL.startswith("https://"), "Cole a URL HTTPS completa."
GITHUB_PAT = getpass("GitHub PAT — Contents: Read/Write, Actions: Read, Secrets: Read/Write: ").strip()
assert GITHUB_PAT, "PAT obrigatório."

%cd /content
REPO_DIR = Path("/content/HookLink2iOS-repo")
if not (REPO_DIR / ".git").is_dir():
    !git clone --depth 1 --branch HookLink2iOS https://github.com/joao9vegeto-stack/KintTany.git /content/HookLink2iOS-repo
else:
    !git -C /content/HookLink2iOS-repo fetch --depth 1 origin HookLink2iOS
    !git -C /content/HookLink2iOS-repo checkout HookLink2iOS
    !git -C /content/HookLink2iOS-repo reset --hard FETCH_HEAD

sys.path.insert(0, str(REPO_DIR / "colab"))
sys.modules.pop("hooklink2ios_prepare", None)
importlib.invalidate_caches()
import hooklink2ios_prepare as hooklink
print("Branch:", BRANCH)
print("Commit do pipeline:", subprocess.check_output(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],text=True).strip())
assert hooklink.BRANCH == BRANCH and hooklink.TITLE_ID == "0400486024DEC000"

## 2. Autorizar o build no GitHub Actions

O runner macOS precisa ler os arquivos privados do Drive. O notebook entrega ao Actions a credencial OAuth já autenticada, criptografada como secret temporário do repositório. Não é service account. O secret é removido ao final da transferência.

## 3. Recuperar uma preparação já concluída

Procura o handoff desta mesma origem na branch e valida os arquivos persistidos no Drive. Se encontrar, não baixa, não extrai e não repete o preparo Linux.

In [ ]:
RECOVERED = hooklink.recover_current_handoff(github_token=GITHUB_PAT, source_url=SOURCE_URL)
HANDOFF_READY = RECOVERED is not None

if HANDOFF_READY:
    HANDOFF = RECOVERED["handoff"]
    hooklink.verify_handoff_drive_assets(drive_api_client, HANDOFF)
    # Republica o handoff para iniciar um novo Actions vinculado a este commit.
    GITHUB_RESULT = hooklink.publish(GITHUB_PAT, HANDOFF)
    HANDOFF_COMMIT_SHA = GITHUB_RESULT["commit_sha"]
    print("Preparação recuperada do Drive; sem novo download ou extração.")
else:
    print("Nenhum handoff válido para esta URL. A próxima etapa fará o download/preparo.")

RECOVERED = hooklink.recover_current_handoff(github_token=GITHUB_PAT, source_url=SOURCE_URL)
HANDOFF_READY = RECOVERED is not None

if HANDOFF_READY:
    HANDOFF = RECOVERED["handoff"]
    hooklink.verify_handoff_drive_assets(drive_api_client, HANDOFF)
    # Republica o handoff para iniciar um novo Actions vinculado a este commit.
    GITHUB_RESULT = hooklink.publish(GITHUB_PAT, HANDOFF)
    HANDOFF_COMMIT_SHA = GITHUB_RESULT["commit_sha"]
    print("Preparação recuperada do Drive; sem novo download ou extração.")
else:
    print("Nenhum handoff válido para esta URL. A próxima etapa fará o download/preparo.")

## 4. Primeira preparação Linux (somente se não houver recuperação)

Na primeira execução baixa a URL fornecida. Se for arquivo compactado, extrai exatamente um NUP, envia o NUP ao Drive e valida o upload antes de apagar o arquivo compacto local. Depois gera RomFS.img e o overlay, envia os arquivos ao Drive e publica o handoff somente na branch HookLink2iOS.

In [ ]:
if HANDOFF_READY:
    print("Preparação já recuperada. Pulando download, extração e geração de RomFS.")
else:
    RESULT = hooklink.prepare(
        source_url=SOURCE_URL,
        drive_parent_folder_id=DRIVE_FOLDER_ID,
        drive=drive_api_client,
        github_token=GITHUB_PAT,
        repo_root=REPO_DIR,
        work_root=ROOT,
    )
    HANDOFF = RESULT["handoff"]
    GITHUB_RESULT = RESULT["github"]
    HANDOFF_COMMIT_SHA = GITHUB_RESULT["commit_sha"]
    print("Download/preparação concluídos. NUP persistido no Drive.")

## 5. Verificar handoff e identificar o build

Confirma Title ID, versão, hashes e commit que deve gerar a IPA. O workflow é restrito à branch HookLink2iOS e usa runner macos-26.

In [ ]:
hooklink.verify_handoff_drive_assets(drive_api_client, HANDOFF)
assert HANDOFF["project"] == "HookLink2iOS"
assert HANDOFF["title_id"] == "0400486024DEC000"
assert HANDOFF["package_version"] == "v131072"
assert HANDOFF["branch"] == BRANCH
assert HANDOFF_COMMIT_SHA
print("Title ID:", HANDOFF["title_id"], "| versão:", HANDOFF["package_version"])
print("NUP:", f'{HANDOFF["nup"]["size"]:,}', "bytes")
print("RomFS:", f'{HANDOFF["romfs"]["size"]:,}', "bytes")
print("Overlay:", f'{HANDOFF["resolution_overlay"]["size"]:,}', "bytes")
print("Commit do build:", HANDOFF_COMMIT_SHA)
print("Commit:", GITHUB_RESULT.get("commit_url"))
print("Actions:", GITHUB_RESULT.get("actions_url"))

## 6. Última etapa: aguardar o Actions e enviar IPA ao Drive

Aguarda somente o workflow do commit acima. Baixa o artifact HookLink2iOS, valida digest e relatório, envia IPA/overlay/relatório para a pasta desta execução no Drive e verifica o upload.

In [ ]:
try:
    TRANSFER = hooklink.transfer_actions_artifact_to_drive(
        github_token=GITHUB_PAT,
        expected_commit=HANDOFF_COMMIT_SHA,
        drive=drive_api_client,
        run_folder_id=HANDOFF["drive"]["run_folder_id"],
    )
finally:
    hooklink.delete_github_actions_secret(GITHUB_PAT, "HOOKLINK2IOS_DRIVE_OAUTH_JSON")

RUN = TRANSFER["run"]
IPA = TRANSFER["ipa"]
print("=" * 64)
print("HOOKLINK2iOS — IPA ENVIADA E VERIFICADA NO GOOGLE DRIVE")
print("=" * 64)
print("Branch:", BRANCH)
print("Commit:", TRANSFER["commit_sha"])
print("Workflow:", RUN["html_url"])
print("IPA:", IPA["name"])
print("Tamanho:", f'{int(IPA["size"]):,}', "bytes")
print("SHA-256:", TRANSFER["ipa_sha256"])
print("Google Drive:", IPA["webViewLink"])
print("Pasta da execução:", HANDOFF["drive"].get("run_folder_url"))
print("Overlay e build-report.json também foram enviados.")